In [5]:
import mne.io
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import time
from zoneinfo import ZoneInfo
import os
import re
from scipy.signal import welch
import glob

dir_path = "D:/research assistant/EEG/EEG_data/EEG_GBL/"
clean_path = "EEG_clean_GBL"
raw_path = "EEG_raw_data_GBL"
processed_path = "Processed GBL Files"

In [ ]:
# rename the clean data files to obey the naming convention for mne.io.read_raw_fif()
for filename in os.listdir(os.path.join(dir_path,"EEG_clean_GBL")):
    new_filename = filename.replace(".fif", "_raw.fif")
    file_path = os.path.join(dir_path, clean_path, filename)
    new_file_path = os.path.join(dir_path, clean_path, new_filename)
    os.rename(file_path, new_file_path)

In [16]:
# this cell is to segment the clean data based on the labelling from processed files
processed_dir = os.path.join(dir_path, processed_path)
segment_dir = os.path.join(dir_path, "segments")

subject_data = {}
for filename in os.listdir(processed_dir):
    processed_file_path = os.path.join(processed_dir, filename)
    # read the separated sheets from processed files
    processed_df_1 = pd.read_excel(processed_file_path, sheet_name="process")
    processed_df_2 = pd.read_excel(processed_file_path, sheet_name="process_group")
    processed_df_3 = pd.read_excel(processed_file_path, sheet_name="process_category")
    # find the index of the subject to locate the related clean file and raw file
    number = re.findall(r'p0*(\d*)', filename)[0]
    if number != "66":
        continue
    try:
        # get the corresponding clean file
        subject_id = str(number).zfill(2)
        pattern = os.path.join(dir_path, clean_path, f"EEG_s{subject_id}*_clean_raw.fif")
        matching_files = glob.glob(pattern)
        if not matching_files:
            raise FileNotFoundError(f"No file found for subject {subject_id} with pattern: {pattern}")
        clean_file = mne.io.read_raw_fif(matching_files[0], preload=True)
        data, times = clean_file[:]
        channel_names = clean_file.ch_names
        clean_df = pd.DataFrame(data.T, columns=channel_names)

        # get the corresponding raw file
        raw_file_path = os.path.join(dir_path, raw_path, f"raw_EEG_s{number}.csv")
        raw_df = pd.read_csv(raw_file_path)
        timestamp = raw_df["Timestamp"]
        clean_df['time'] = timestamp
    except Exception as e:
        print(e)
    segments = []
    output_path = os.path.join(segment_dir, number)    
    for index, row in processed_df_1.iterrows():
        # convert the epoch time to local time zone
        start = row['start_time'].replace(tzinfo=ZoneInfo("Asia/Singapore")).timestamp()
        end = row['end_time'].replace(tzinfo=ZoneInfo("Asia/Singapore")).timestamp()
        label_1 = row['process']
        label_2 = processed_df_2.iloc[index]['process_group']
        label_3 = processed_df_3.iloc[index]['process_category']
        segment = clean_df[(clean_df['time'] >= start) & (clean_df['time'] < end + 1)]
        if not segment.empty:
            segment['process'] = label_1
            segment['process_group'] = label_2
            segment['process_category'] = label_3
            segments.append(segment)
            if not os.path.exists(output_path):
                os.makedirs(output_path)
            file_name = row['start_time'].strftime("%Y-%m-%d %H-%M-%S") + "_" + row['end_time'].strftime("%Y-%m-%d %H-%M-%S") + ".xlsx"
            segment.to_excel(os.path.join(output_path, file_name))
    subject_data[number] = segments

Opening raw data file D:/research assistant/EEG/EEG_data/EEG_GBL/EEG_clean_GBL\EEG_s66_clean_raw.fif...
    Range : 0 ... 356632 =      0.000 ...  2786.188 secs
Ready.
Reading 0 ... 356632  =      0.000 ...  2786.188 secs...


C:\Users\jiang_c\AppData\Local\Temp\ipykernel_25304\3524721361.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process'] = label_1
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_25304\3524721361.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process_group'] = label_2
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_25304\3524721361.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

In [9]:
clean_numbers = []
for file in os.listdir(os.path.join(dir_path, clean_path)):
    clean_numbers.append(re.findall(r's0*(\d*)', file)[0])
processed_numbers = []
for file in os.listdir(os.path.join(dir_path, processed_path)):
    processed_numbers.append(re.findall(r'p0*(\d*)', file)[0])

In [12]:
len(clean_numbers) - len(processed_numbers)

4

In [13]:
list(set(clean_numbers) - set(processed_numbers))

['65', '60', '64', '54', '104']

In [14]:
list(set(processed_numbers) - set(clean_numbers))

['37']